# ???????? DETR ??? ???????? ???????? ?? ???

??????? ??? Google Colab. ?????????????? ???????, ??????? DETR ? ????????? ????, ??????? ? ??????? ?? Google Drive.


In [ ]:
!pip install -q kagglehub torchmetrics pycocotools transformers timm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PROJECT_NAME = 'brain-mri-detr'
WORKDIR = Path('/content') / PROJECT_NAME
ARTIFACTS_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
DATASET_DIR = WORKDIR / 'dataset'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
RUNS_DIR = ARTIFACTS_DIR / 'runs'

EPOCHS = 3
BATCH_SIZE = 2
NUM_WORKERS = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
SEED = 42
SCORE_THRESHOLD = 0.5
TRAIN_FRACTION = 0.35
VAL_FRACTION = 0.5
IMAGE_SIZE = 512

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
NUM_CLASSES = len(CLASS_NAMES)
ID2LABEL = {i: name for i, name in enumerate(CLASS_NAMES)}
LABEL2ID = {name: i for i, name in enumerate(CLASS_NAMES)}

WORKDIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('WORKDIR:', WORKDIR)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)


In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
import kagglehub
import shutil
from pathlib import Path

path = kagglehub.dataset_download('ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes')
dataset_path = Path(path)
print('Raw kagglehub path:', dataset_path)

candidate_roots = [dataset_path] + [p for p in dataset_path.rglob('*') if p.is_dir()]
resolved_root = None
for candidate in candidate_roots:
    if (candidate / 'Train').exists() and (candidate / 'Val').exists():
        resolved_root = candidate
        break

if resolved_root is None:
    raise FileNotFoundError('?? ??????? ????? ????? Train/Val ?????? ????????.')

dataset_target = DATASET_DIR / 'brain-tumor-mri'
if dataset_target.exists():
    shutil.rmtree(dataset_target)
shutil.copytree(resolved_root, dataset_target)

print('Resolved dataset root:', resolved_root)
print('Copied dataset to:', dataset_target)


In [ ]:
from pathlib import Path
import random

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import DetrImageProcessor

processor = DetrImageProcessor.from_pretrained(
    'facebook/detr-resnet-50',
    do_resize=True,
    size={'shortest_edge': IMAGE_SIZE, 'longest_edge': IMAGE_SIZE},
)

class BrainMRIDetrDataset(Dataset):
    def __init__(self, dataset_root, split_name, class_names, drop_empty=True):
        self.dataset_root = Path(dataset_root)
        self.split_name = split_name
        self.class_names = class_names
        self.drop_empty = drop_empty
        self.records = []
        split_dir = self.dataset_root / split_name

        skipped_missing = 0
        skipped_empty = 0
        skipped_invalid = 0

        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            for image_path in sorted((class_dir / 'images').glob('*')):
                label_path = class_dir / 'labels' / f'{image_path.stem}.txt'
                if not label_path.exists():
                    skipped_missing += 1
                    if drop_empty:
                        continue

                valid_boxes = 0
                if label_path.exists():
                    lines = label_path.read_text(encoding='utf-8').strip().splitlines()
                    if not lines:
                        skipped_empty += 1
                        if drop_empty:
                            continue
                    for line in lines:
                        if not line.strip():
                            continue
                        parts = line.split()
                        if len(parts) != 5:
                            continue
                        try:
                            class_id, x_center, y_center, box_width, box_height = map(float, parts)
                        except ValueError:
                            continue
                        if box_width <= 0 or box_height <= 0:
                            continue
                        if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
                            continue
                        if not (0 < box_width <= 1 and 0 < box_height <= 1):
                            continue
                        valid_boxes += 1

                if drop_empty and valid_boxes == 0:
                    skipped_invalid += 1
                    continue

                self.records.append({
                    'image_path': image_path,
                    'label_path': label_path,
                })

        print(f'{split_name}: kept={len(self.records)}, skipped_missing={skipped_missing}, skipped_empty={skipped_empty}, skipped_invalid={skipped_invalid}')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record['image_path']).convert('RGB')
        width, height = image.size

        annotations = []
        if record['label_path'].exists():
            content = record['label_path'].read_text(encoding='utf-8').strip().splitlines()
            for ann_idx, line in enumerate(content):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    continue
                class_id, x_center, y_center, box_width, box_height = map(float, parts)
                if box_width <= 0 or box_height <= 0:
                    continue

                x1 = (x_center - box_width / 2.0) * width
                y1 = (y_center - box_height / 2.0) * height
                w = box_width * width
                h = box_height * height

                x1 = max(0.0, min(x1, width - 1))
                y1 = max(0.0, min(y1, height - 1))
                w = max(1.0, min(w, width - x1))
                h = max(1.0, min(h, height - y1))

                annotations.append({
                    'id': ann_idx,
                    'image_id': index,
                    'category_id': int(class_id),
                    'bbox': [x1, y1, w, h],
                    'area': w * h,
                    'iscrowd': 0,
                })

        target = {
            'image_id': index,
            'annotations': annotations,
        }

        encoded = processor(images=image, annotations=target, return_tensors='pt')
        pixel_values = encoded['pixel_values'].squeeze(0)
        labels = encoded['labels'][0]
        return pixel_values, labels


def collate_fn(batch):
    pixel_values = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    encoding = processor.pad(pixel_values, return_tensors='pt')
    return {
        'pixel_values': encoding['pixel_values'],
        'pixel_mask': encoding['pixel_mask'],
        'labels': labels,
    }


def make_subset(dataset, fraction, seed=42):
    if fraction >= 1.0:
        return dataset
    size = len(dataset)
    subset_size = max(1, int(size * fraction))
    rng = random.Random(seed)
    indices = list(range(size))
    rng.shuffle(indices)
    return Subset(dataset, indices[:subset_size])

DATA_ROOT = DATASET_DIR / 'brain-tumor-mri'

full_train_dataset = BrainMRIDetrDataset(DATA_ROOT, 'Train', CLASS_NAMES, drop_empty=True)
full_val_dataset = BrainMRIDetrDataset(DATA_ROOT, 'Val', CLASS_NAMES, drop_empty=True)

train_dataset = make_subset(full_train_dataset, TRAIN_FRACTION, SEED)
val_dataset = make_subset(full_val_dataset, VAL_FRACTION, SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print('Full train samples:', len(full_train_dataset))
print('Full val samples:', len(full_val_dataset))
print('Used train samples:', len(train_dataset))
print('Used val samples:', len(val_dataset))


In [ ]:
import json

dataset_summary = {
    'full_train_samples': len(full_train_dataset),
    'full_val_samples': len(full_val_dataset),
    'used_train_samples': len(train_dataset),
    'used_val_samples': len(val_dataset),
    'train_fraction': TRAIN_FRACTION,
    'val_fraction': VAL_FRACTION,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
}

summary_path = ARTIFACTS_DIR / 'dataset_summary.json'
summary_path.write_text(json.dumps(dataset_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))


In [ ]:
from transformers import DetrForObjectDetection

model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(DEVICE)
model


In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

history = []
best_map50 = -1.0
best_weights_path = RUNS_DIR / 'best_detr.pt'
last_weights_path = RUNS_DIR / 'last_detr.pt'


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    valid_steps = 0

    for batch in tqdm(loader, desc='Train', leave=False):
        pixel_values = batch['pixel_values'].to(device)
        pixel_mask = batch['pixel_mask'].to(device)
        labels = [{k: v.to(device) for k, v in label.items()} for label in batch['labels']]

        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        loss = outputs.loss

        if not torch.isfinite(loss):
            print('Non-finite loss detected, skipping step')
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        valid_steps += 1

    return running_loss / max(valid_steps, 1)


def evaluate_model(model, loader, device, score_threshold=0.5):
    metric = MeanAveragePrecision(class_metrics=False)
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc='Val', leave=False):
            pixel_values = batch['pixel_values'].to(device)
            pixel_mask = batch['pixel_mask'].to(device)
            labels = batch['labels']

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
            target_sizes = torch.stack([label['orig_size'] for label in labels]).to(device)
            results = processor.post_process_object_detection(outputs, threshold=score_threshold, target_sizes=target_sizes)

            preds = []
            refs = []
            for result, label in zip(results, labels):
                preds.append({
                    'boxes': result['boxes'].detach().cpu(),
                    'scores': result['scores'].detach().cpu(),
                    'labels': result['labels'].detach().cpu(),
                })

                gt_boxes = label['boxes'].detach().cpu()
                gt_boxes_xyxy = torch.zeros_like(gt_boxes)
                gt_boxes_xyxy[:, 0] = gt_boxes[:, 0] - gt_boxes[:, 2] / 2.0
                gt_boxes_xyxy[:, 1] = gt_boxes[:, 1] - gt_boxes[:, 3] / 2.0
                gt_boxes_xyxy[:, 2] = gt_boxes[:, 0] + gt_boxes[:, 2] / 2.0
                gt_boxes_xyxy[:, 3] = gt_boxes[:, 1] + gt_boxes[:, 3] / 2.0

                h, w = label['orig_size'].detach().cpu()
                scale = torch.tensor([w, h, w, h], dtype=torch.float32)
                gt_boxes_xyxy = gt_boxes_xyxy * scale

                refs.append({
                    'boxes': gt_boxes_xyxy,
                    'labels': label['class_labels'].detach().cpu(),
                })

            metric.update(preds, refs)

    computed = metric.compute()
    return {
        'map': float(computed['map']),
        'map_50': float(computed['map_50']),
        'mar_100': float(computed['mar_100']),
    }


for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, DEVICE, SCORE_THRESHOLD)
    lr_scheduler.step()

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'map': val_metrics['map'],
        'map_50': val_metrics['map_50'],
        'mar_100': val_metrics['mar_100'],
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)
    print(row)

    torch.save(model.state_dict(), last_weights_path)
    if row['map_50'] > best_map50:
        best_map50 = row['map_50']
        torch.save(model.state_dict(), best_weights_path)

history_df = pd.DataFrame(history)
history_path = RUNS_DIR / 'metrics.csv'
history_df.to_csv(history_path, index=False)
history_df


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['epoch'], history_df['train_loss'], marker='o')
plt.title('Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_df['epoch'], history_df['map_50'], marker='o', label='mAP@50')
plt.plot(history_df['epoch'], history_df['map'], marker='o', label='mAP@50:95')
plt.title('Validation Metrics')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.tight_layout()

metrics_plot_path = RUNS_DIR / 'training_curves.png'
plt.savefig(metrics_plot_path, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import functional as F
from transformers import DetrForObjectDetection

best_model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
best_model.load_state_dict(torch.load(best_weights_path, map_location=DEVICE))
best_model.to(DEVICE)
best_model.eval()

sample_count = 6
base_val_dataset = val_dataset.dataset if hasattr(val_dataset, 'dataset') else val_dataset
subset_indices = val_dataset.indices if hasattr(val_dataset, 'indices') else list(range(len(base_val_dataset)))
selected_indices = subset_indices[:min(sample_count, len(subset_indices))]

preview_dir = PREDICTIONS_DIR / 'val_preview'
preview_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(14, 10))
with torch.no_grad():
    for plot_idx, dataset_idx in enumerate(selected_indices, start=1):
        record = base_val_dataset.records[dataset_idx]
        image_path = record['image_path']
        image = Image.open(image_path).convert('RGB')

        inputs = processor(images=image, return_tensors='pt')
        pixel_values = inputs['pixel_values'].to(DEVICE)
        pixel_mask = inputs['pixel_mask'].to(DEVICE)
        outputs = best_model(pixel_values=pixel_values, pixel_mask=pixel_mask)

        target_sizes = torch.tensor([[image.size[1], image.size[0]]], device=DEVICE)
        result = processor.post_process_object_detection(outputs, threshold=SCORE_THRESHOLD, target_sizes=target_sizes)[0]

        image_np = np.array(image).copy()
        boxes = result['boxes'].detach().cpu().numpy()
        labels = result['labels'].detach().cpu().numpy()
        scores = result['scores'].detach().cpu().numpy()

        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.astype(int)
            class_name = CLASS_NAMES[int(label)]
            cv2.rectangle(image_np, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image_np, f'{class_name} {score:.2f}', (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        output_path = preview_dir / f'image{plot_idx - 1}.jpg'
        Image.fromarray(image_np).save(output_path)

        plt.subplot(2, 3, plot_idx)
        plt.imshow(image_np)
        plt.axis('off')
        plt.title(image_path.name)

plt.tight_layout()
plt.show()


In [ ]:
import shutil
import json

final_dir = ARTIFACTS_DIR / 'final_artifacts'
final_dir.mkdir(parents=True, exist_ok=True)

if len(history_df) > 0:
    best_row_idx = history_df['map_50'].idxmax()
    summary_metrics = history_df.loc[best_row_idx].to_dict()
else:
    summary_metrics = {}

summary_metrics_path = final_dir / 'best_metrics.json'
summary_metrics_path.write_text(json.dumps(summary_metrics, ensure_ascii=False, indent=2), encoding='utf-8')

files_to_copy = [
    best_weights_path,
    last_weights_path,
    history_path,
    metrics_plot_path,
    summary_path,
]

for file_path in files_to_copy:
    if file_path.exists():
        destination = final_dir / file_path.name
        if file_path.resolve() != destination.resolve():
            shutil.copy2(file_path, destination)

preview_dir = PREDICTIONS_DIR / 'val_preview'
final_preview_dir = final_dir / 'val_preview'

if preview_dir.exists():
    if final_preview_dir.exists():
        shutil.rmtree(final_preview_dir)
    shutil.copytree(preview_dir, final_preview_dir)

print('Artifacts directory:', final_dir)
print('Available files:')
for path in sorted(final_dir.glob('*')):
    print('-', path.name)


## ??? ????????? ????? ????????

????? ?????????? ???????? ??????? ????????? `best_detr.pt`, `last_detr.pt`, `metrics.csv`, `training_curves.png`, `dataset_summary.json` ? `best_metrics.json`.
